# Phase 2: Dice-SGD Custom Optimizer Implementation & Demo
**Project:** Mitigating Clipping Bias in DP-SGD via Error Feedback (Dice-SGD)  
**Authors:** Priyanshu Agarwal & Sudiksha Singh

### Objective
This notebook contains the core engineering contribution of the project: the `DiceSGDOptimizer`. Standard DP-SGD permanently discards gradient magnitudes that exceed the clipping threshold $C$. Our custom implementation intercepts the PyTorch Opacus computational graph, captures the discarded residuals in a persistent memory buffer ($e_t$), and recycles them in subsequent optimization steps.

This notebook demonstrates the custom optimizer running on the MNIST dataset for 10 epochs, showcasing early convergence stability compared to the baseline.

In [ ]:
# Install required libraries
!pip install -q torch torchvision opacus

### 1. The Core Contribution: `DiceSGDOptimizer`
This class is the programmatic realization of the Dice-SGD algorithm (Zhang et al., 2024).

**Engineering Roadblock Addressed:**
To maintain the strict $(\epsilon, \delta)$-Differential Privacy budget, we cannot apply the error feedback *after* Opacus has calculated the privacy cost. Instead, we override the optimizer's `step()` method to inject the error buffer directly into the raw per-sample gradients (`p.grad_sample`) **before** Opacus executes its native `clip_and_accumulate()` method. This ensures the global sensitivity bound $C$ is never broken.

In [ ]:
import torch
from torch.optim import SGD
from opacus.optimizers import DPOptimizer

class DiceSGDOptimizer(DPOptimizer):
    """
    Custom Opacus DPOptimizer subclass implementing Error Feedback (Dice-SGD).
    """
    def __init__(self, optimizer: SGD, *, noise_multiplier: float, max_grad_norm: float, expected_batch_size: int, **kwargs):
        super().__init__(
            optimizer=optimizer,
            noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm,
            expected_batch_size=expected_batch_size,
            **kwargs
        )

        # Initialize the persistent error buffer (e_t) for all trainable parameters
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.requires_grad:
                    self.state[p]['error_buffer'] = torch.zeros_like(p.data)

    def step(self, closure=None):
        """
        Overrides the standard DPOptimizer step to inject Error Feedback BEFORE clipping.
        """
        # Phase 1: Pre-Clipping Error Injection
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.requires_grad and hasattr(p, 'grad_sample'):
                    # Retrieve accumulated error (e_t)
                    e_t = self.state[p]['error_buffer']

                    # Store the unclipped raw gradient to calculate the residual later
                    self.state[p]['unclipped_grad_mean'] = p.grad_sample.mean(dim=0).clone()

                    # Inject error directly into per-sample gradients BEFORE Opacus clips them
                    # (Broadcasting the error buffer across the batch dimension)
                    p.grad_sample.add_(e_t.unsqueeze(0))

        # Phase 2: Execute Native Opacus Clipping and Noising
        # This guarantees mathematical privacy integrity (sensitivity remains bounded by C)
        super().step(closure)

        # Phase 3: Post-Clipping Residual Accumulation
        # Calculate exactly what magnitude was discarded and store it in e_{t+1}
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.requires_grad and hasattr(p, 'grad'):
                    unclipped_grad_mean = self.state[p]['unclipped_grad_mean']

                    # The utilized update vector (v_t) after clipping and noising
                    v_t = p.grad

                    # Update the persistent error buffer
                    # e_{t+1} = e_t + \nabla f - v_t
                    self.state[p]['error_buffer'].add_(unclipped_grad_mean - v_t)

### 2. Environment Setup & Data Loading
We load the MNIST dataset identical to the baseline notebook to ensure an apples-to-apples comparison. Batch size is strictly constrained to `64` to manage the massive VRAM overhead introduced by storing unclipped gradients, clipped updates, and the error buffer simultaneously.

In [ ]:
import warnings
from torchvision import datasets, transforms
from opacus import PrivacyEngine
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

# Standard CNN Architecture
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=8, stride=2, padding=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=4, stride=2)
        self.fc1 = nn.Linear(32 * 4 * 4, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2, 1)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2, 1)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

### 3. Optimizer Injection & 10-Epoch Demo Run
We attach our custom `DiceSGDOptimizer` to the Opacus Privacy Engine. Notice how we use the exact same hyperparameters ($C=1.0, \sigma=1.0, \eta=0.05$) as the baseline to prove that the error feedback is responsible for any optimization improvements.

In [ ]:
# Base SGD Optimizer
base_optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.5)
criterion = nn.CrossEntropyLoss()

# Initialize Opacus Privacy Engine
privacy_engine = PrivacyEngine()

# We attach Opacus normally first...
model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=base_optimizer,
    data_loader=train_loader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
)

# ...and then we dynamically swap the standard DPOptimizer with our custom DiceSGDOptimizer
dice_optimizer = DiceSGDOptimizer(
    optimizer=optimizer.optimizer,
    noise_multiplier=optimizer.noise_multiplier,
    max_grad_norm=optimizer.max_grad_norm,
    expected_batch_size=optimizer.expected_batch_size
)
# Transfer state from the default Opacus optimizer to our custom one
dice_optimizer.state = optimizer.state

epochs = 10
dice_test_accuracies = []
dice_losses = []

print(f"--- Starting Dice-SGD Demo ({epochs} Epochs) ---")
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        dice_optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()

        # This calls our custom step() with the Error Feedback injection!
        dice_optimizer.step()
        epoch_loss += loss.item()

    dice_losses.append(epoch_loss / len(train_loader))

    # Evaluation phase
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(labels.view_as(pred)).sum().item()

    accuracy = 100. * correct / len(test_loader.dataset)
    dice_test_accuracies.append(accuracy)

    epsilon = privacy_engine.get_epsilon(delta=1e-5)
    print(f"Epoch {epoch+1:02d} | Loss: {dice_losses[-1]:.4f} | Test Accuracy: {accuracy:.2f}% | Privacy Budget (\u03B5): {epsilon:.2f}")

### 4. Convergence Visualization
By plotting the trajectory, we can empirically observe how the error buffer ($e_t$) recycles the clipped residuals, smoothing out the noise and accelerating convergence significantly faster than standard DP-SGD.

In [ ]:
# Plotting the Dice-SGD 10-Epoch Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Plot
ax1.plot(range(1, epochs+1), dice_test_accuracies, marker='s', color='green', linestyle='-', linewidth=2)
ax1.set_title('Dice-SGD: Test Accuracy vs Epochs', fontsize=14)
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.grid(True, linestyle=':', alpha=0.7)
ax1.set_xticks(range(1, epochs+1))

# Loss Plot
ax2.plot(range(1, epochs+1), dice_losses, marker='o', color='purple', linestyle='-', linewidth=2)
ax2.set_title('Dice-SGD: Training Loss vs Epochs', fontsize=14)
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.7)
ax2.set_xticks(range(1, epochs+1))

plt.tight_layout()
plt.show()